# Bayesian A/B Engine — Exploration Notebook

This notebook walks through each module step-by-step: data simulation, validation, Bayesian inference, novelty detection, and posterior analysis.


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az

from data.simulate_events import simulate_experiment
from data.loader import load_from_dataframe
from engine.validator import validate_experiment
from engine.bayesian_model import prepare_inputs, run_bayesian_ab_test
from engine.posterior_analysis import analyse_posterior
from engine.novelty_detector import detect_novelty_effect

## 1. Simulate experiment data

In [ ]:
df = simulate_experiment(
    n_users=10_000,
    true_control_cvr=0.05,
    true_treatment_cvr=0.065,
    novelty_boost=0.04,
    novelty_decay_days=5,
    experiment_days=21,
)
print(df.shape)
df.groupby('variant')[['converted']].agg(['count','mean']).round(4)

## 2. Validate the experiment (SRM check)

In [ ]:
validation = validate_experiment(df)
print(f'SRM detected: {validation.srm_detected} (p={validation.srm_p_value:.4f})')
print(f'Control n={validation.control_n}, Treatment n={validation.treatment_n}')
for w in validation.warnings:
    print('WARNING:', w)

## 3. Run the Bayesian model

In [ ]:
inputs = prepare_inputs(df)
print(inputs)

trace = run_bayesian_ab_test(inputs, draws=1000, tune=500, chains=2)

In [ ]:
# Posterior diagnostics
az.plot_trace(trace, var_names=['p_control', 'p_treatment', 'lift'])
plt.tight_layout()
plt.show()

## 4. Posterior analysis

In [ ]:
summary = analyse_posterior(trace, annual_baseline_gmv=10_000_000)
print(f'P(B > A):          {summary.prob_b_beats_a:.1%}')
print(f'Lift (mean):       {summary.lift_mean:.3%}')
print(f'HDI 95%:           {summary.lift_hdi_lower:.3%} – {summary.lift_hdi_upper:.3%}')
print(f'Relative lift:     {summary.rel_lift_mean:.1%}')
print(f'ROPE overlap:      {summary.rope_overlap:.1%}')
if summary.revenue_low:
    print(f'Revenue impact:    ${summary.revenue_low:,.0f} – ${summary.revenue_high:,.0f}')

In [ ]:
# Plot posterior distribution of lift
lift_samples = trace.posterior['lift'].values.flatten()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(lift_samples * 100, bins=80, color='steelblue', alpha=0.7, density=True)
ax.axvline(0, color='red', linestyle='--', linewidth=1, label='No effect')
ax.axvline(summary.lift_hdi_lower * 100, color='orange', linestyle=':', label='HDI lower')
ax.axvline(summary.lift_hdi_upper * 100, color='orange', linestyle=':', label='HDI upper')
ax.set_xlabel('Absolute lift (%)')
ax.set_ylabel('Posterior density')
ax.set_title(f'Posterior lift distribution  |  P(B>A) = {summary.prob_b_beats_a:.1%}')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Novelty effect detection

In [ ]:
novelty = detect_novelty_effect(df)
print(f'Risk level:     {novelty.risk_level}')
print(f'Novelty score:  {novelty.novelty_score}')
print(f'Slope:          {novelty.slope:.6f}/day  (p={novelty.slope_p_value:.4f})')
print(f'Recommendation: {novelty.recommendation}')

In [ ]:
# Plot daily lift over time
days  = [d['day']  for d in novelty.daily_lifts]
lifts = [d['lift'] * 100 for d in novelty.daily_lifts]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(days, lifts, marker='o', color='orange', linewidth=2, markersize=5)
ax.axhline(0, color='gray', linestyle='--', linewidth=1)

# Trend line
x = np.array(days)
trend = novelty.slope * x * 100 + (lifts[0] - novelty.slope * days[0] * 100)
ax.plot(days, trend, color='red', linestyle='--', linewidth=1.5, label='Trend')

ax.set_xlabel('Experiment day')
ax.set_ylabel('Lift (%)')
ax.set_title(f'Daily lift timeline  |  Novelty risk: {novelty.risk_level}  (score {novelty.novelty_score})')
ax.legend()
plt.tight_layout()
plt.show()